# Reading and filtering
The cheapest kind of work there is: opening a file and looking at part of it.

These cases barely do any computing at all, so what they mostly measure is **overhead** —
the fixed cost of planning a job, splitting it into tasks and scheduling them. On a single
machine that cost is paid without any of the benefit it buys on a cluster.

Watch the difference between **S3** (three columns) and **S4** (all twenty-two). Parquet
stores each column separately, like a filing cabinet with one drawer per column. A good
engine opens only the drawers it needs. S4 is what happens when you make it tip out the
whole cabinet.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "01_scan_and_filter", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")

In [ ]:
SQL_S1 = """
SELECT count(*) AS n FROM sales
"""

# WHY THIS ONE:
#   The cheapest possible question. Mostly measures overhead, not work.
_, out, _ = bench.run(Case("S1", "Count all rows", "Scan", sql=SQL_S1.strip(),
                          why="The cheapest possible question. Mostly measures overhead, not work."))
display(out.head())

In [ ]:
SQL_S2 = """
SELECT count(*) AS n, round(sum(amount), 2) AS total
FROM sales
WHERE amount > 280 AND channel = 'app' AND region = 'north'
"""

# WHY THIS ONE:
#   Can the engine skip most of the file instead of reading it all?
_, out, _ = bench.run(Case("S2", "Selective filter (~1% of rows)", "Scan", sql=SQL_S2.strip(),
                          why="Can the engine skip most of the file instead of reading it all?"))
display(out.head())

In [ ]:
SQL_S3 = """
SELECT channel, round(sum(amount), 2) AS total, sum(quantity) AS units
FROM sales GROUP BY 1 ORDER BY 1
"""

# WHY THIS ONE:
#   Column pruning: open three drawers of the cabinet, not all 22.
_, out, _ = bench.run(Case("S3", "Read 3 columns out of 22", "Scan", sql=SQL_S3.strip(),
                          why="Column pruning: open three drawers of the cabinet, not all 22."))
display(out.head())

In [ ]:
SQL_S4 = """
SELECT count(*) AS n, count(session_ref) AS a, count(user_agent) AS b,
       count(attributes_json) AS c, count(notes) AS d, count(promo_code) AS e,
       round(sum(amount),2) AS f, sum(quantity) AS g, round(sum(unit_price),2) AS h,
       round(sum(discount_pct),2) AS i, round(sum(tax_pct),2) AS j,
       count(DISTINCT channel) AS k, count(DISTINCT device) AS l,
       count(DISTINCT payment_method) AS m, count(DISTINCT currency) AS n2,
       count(DISTINCT region) AS o, count(DISTINCT store_id) AS p,
       count(DISTINCT product_id) AS q, min(sale_date) AS r, max(sale_ts) AS s,
       sum(used_promo) AS t, min(sale_id) AS u
FROM sales
"""

# WHY THIS ONE:
#   The opposite of S3 — the whole table. Compare with S3 to see pruning pay off.
_, out, _ = bench.run(Case("S4", "Read all 22 columns", "Scan", sql=SQL_S4.strip(),
                          why="The opposite of S3 — the whole table. Compare with S3 to see pruning pay off."))
display(out.head())

In [ ]:
SQL_S5 = """
SELECT count(*) AS n, round(sum(amount), 2) AS total
FROM sales_part
WHERE sale_month = DATE '2024-06-01'
"""

# WHY THIS ONE:
#   Skipping whole folders on disk rather than filtering rows.
_, out, _ = bench.run(Case("S5", "Partition pruning (one month)", "Scan", sql=SQL_S5.strip(),
                          why="Skipping whole folders on disk rather than filtering rows."))
display(out.head())

## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()